# FiftyOne Demo: Image Classification with InceptionV3

This notebook demonstrates how to:

* Load the flowers classification dataset into FiftyOne
* Create splits for training, validation and testing
* Download the media to the FiftyOne Media Cache ONLY ONCE
* Export ONLY labels for the FiftyOne dataset, without exporting and duplicating
media unnecessarily
* Train an InceptionV3 model to classify over the 5 classes of flowers
* Save the model weights
* Apply the trained model on the test set of images that already exist in the 
FiftyOne Media Cache
* Write the resulting predictions to a manifest, then ingest those labels as
predictions back into the original dataset, without redundant export of media or
extra copying. 

In [ ]:
from pathlib import Path
import json
import time
from typing import List, Tuple

import fiftyone as fo
import fiftyone.utils.random as four

from PIL import Image
from tqdm import tqdm
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, models, transforms

### Note: Media Cache Size

Ensure that the FiftyOne Media Cache size is larger than the dataset size. Note
that the default is 32 GB. 

In [ ]:
fo.media_cache_config.cache_size_bytes=34359738368 #default is 32GB
# ensure the media cache config is large enough to hold the whole dataset

In [ ]:
DATASET_DIR='gs://voxel51-test/dwiref/flowers'
# Replace dataset directory with path to Azure where dataset is saved
# Alternatively, if the dataset is stored locally, replace with local path

dataset_name="flowers_classification_dataset"

device="cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Ingest dataset as type ImageClassificationDirectoryTree
try:
    dataset = fo.Dataset.from_dir(
        dataset_type=fo.types.ImageClassificationDirectoryTree,
        dataset_dir=DATASET_DIR,
        name=dataset_name,
        persistent=True
    )
except ValueError:
    print(f'A dataset with the name {dataset_name} already exists! \n',
          f'You can load the existing dataset instead of creating a new one or,',
          f'delete the existing dataset and try again.')

# If the dataset already exists, you can load it by uncommenting this line:
# dataset = fo.load_dataset(dataset_name)

# compute metadata for performance
dataset.compute_metadata()

# Load classes and their count
flowers_classes = dataset.default_classes
num_classes = len(flowers_classes)
print(f'There are {num_classes} in this FiftyOne dataset. The classes are:',
      f'{flowers_classes}')

In [ ]:
# Confirm media cache size, then download dataset media to local cache
print(f'FiftyOne Cloud Media Cache Size in Bytes:',
      f'{fo.media_cache_config.cache_size_bytes}')

# Note: this will only work on a cloud-backed dataset
# Skip this step if DATASET_DIR is a local path
dataset.download_media()

# Print local filepath for first sample in dataset after cache download
print(f'{dataset.first().local_path}')
# /Users/dwiref/fiftyone/__cache__/media/gcs/voxel51-test/

To avoid exporting the dataset separately for training, we will treat the local
media cache directory as the root dataset directory for model training.

In [ ]:
# Create train/test/validation splits

four.random_split(dataset, {"train": 0.7, "test": 0.2, "val": 0.1})
print(dataset.count_sample_tags())

In [ ]:
# Replace with path to root directory of the dataset in local media cache
LOCAL_DATASET_DIR = Path('/Users/dwiref/fiftyone/__cache__/media/gcs/voxel51-test/dwiref/flowers')

# Parameters
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-3

In [ ]:
train = dataset.match_tags('train')
val = dataset.match_tags('val')
test = dataset.match_tags('test')

In [ ]:
# Export a manifest of filepaths with export_media='manifest' to avoid
# exporting media and creating unwanted redundancy

# Replace the manifest pattern path with any local path where you want to
# save the exported labels. Note, this will NOT export or copy images or media
# but only the filenames and their ground truth labels
MANIFEST_PATTERN = f'/Users/dwiref/Downloads/flowers-'
train.export(
    export_dir=MANIFEST_PATTERN+'train',
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    export_media='manifest',
    classes=dataset.default_classes,
    label_field='ground_truth',
    pretty_print=True
)

val.export(
    export_dir=MANIFEST_PATTERN+'val',
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    export_media='manifest',
    classes=dataset.default_classes,
    label_field='ground_truth',
    pretty_print=True
)

test.export(
    export_dir=MANIFEST_PATTERN+'test',
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    export_media='manifest',
    classes=dataset.default_classes,
    label_field='ground_truth',
    pretty_print=True
)

In [ ]:
MANIFEST_PATH_TRAIN = MANIFEST_PATTERN+'train/labels.json'
MANIFEST_PATH_VAL = MANIFEST_PATTERN+'val/labels.json'
MANIFEST_PATH_TEST = MANIFEST_PATTERN+'test/labels.json'

# 1) --- load the manifests ---
with open(MANIFEST_PATH_TRAIN, 'r') as file:
    manifest_train = json.load(file)

with open(MANIFEST_PATH_VAL, 'r') as file:
    manifest_val = json.load(file)

with open(MANIFEST_PATH_TEST, 'r') as file:
    manifest_test = json.load(file)

idx2class: List[str] = manifest_train["classes"]
class2idx = {c: i for i, c in enumerate(idx2class)}
# {'daisy': 0, 'dandelion': 1, 'roses': 2, 'sunflowers': 3, 'tulips': 4}

def idx_to_subfolder(idx: int) -> str:
    """Map label index to the sub-folder name (“sunflowers”, …)"""
    return idx2class[idx]

# 2) --- build list of (path, idx) for training and validation ---
# train
train_sample_images: List[Tuple[Path, int]] = []
for base_name, idx in manifest_train["labels"].items():
    path = Path(LOCAL_DATASET_DIR / idx_to_subfolder(idx) / f"{base_name}.jpg")
    train_sample_images.append((path, idx))

# val
val_sample_images: List[Tuple[Path, int]] = []
for base_name, idx in manifest_val["labels"].items():
    path = Path(LOCAL_DATASET_DIR / idx_to_subfolder(idx) / f"{base_name}.jpg")
    val_sample_images.append((path, idx))

# test - create a list of image paths to predict on
test_sample_images: List[Path] = []
for base_name, idx in manifest_test["labels"].items():
    path = Path(LOCAL_DATASET_DIR / idx_to_subfolder(idx) / f"{base_name}.jpg")
    test_sample_images.append(path)

# 3) --- tiny custom Dataset around a list ---
class FlowerDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform
    
    def __len__(self): return len(self.items)
    
    def __getitem__(self, i):
        img_path, label = self.items[i]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# Augmentations & preprocessing recommended for InceptionV3
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(299),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

val_tf = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

In [ ]:
train_ds = FlowerDataset(train_sample_images, train_tf)
val_ds = FlowerDataset(val_sample_images, val_tf)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
# Load InceptionV3 with default pretrained weights
model = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT)

# Replace output layer with 5-unit classifier
model.fc = nn.Linear(model.fc.in_features, len(idx2class))
model = model.to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(model.parameters(), lr=LR)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
def unpack_inception(out):
    """
    Returns (logits, aux_logits_or_None) for both training and eval.
    """
    # in eval mode, use only only main head
    if isinstance(out, torch.Tensor):
        return out, None
    # in training mode, you get a tuple
    else:
        return out.logits, out.aux_logits

In [ ]:
def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            # may be Tensor or tuple
            out = model(X)
            logits, aux = unpack_inception(out)
            loss = criterion(logits, y)
            # only when aux present
            if train and aux is not None:
                loss += 0.4 * criterion(aux, y)
            if train:
                loss.backward()
                optimizer.step()
        running_loss += loss.item() * X.size(0)
        preds = logits.argmax(1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
    return running_loss / total, correct / total


In [ ]:
for epoch in range(1, EPOCHS+1):
    t0 = time.perf_counter()
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(val_loader,   train=False)
    scheduler.step()
    dt = time.perf_counter() - t0
    print(f"[{epoch:02d}/{EPOCHS}] "
          f"train loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val loss={val_loss:.4f} acc={val_acc:.3f} | "
          f"{dt:.1f}s")

In [ ]:
# Save model weights
# Replace the output path for where you would like to save these weights
torch.save(
    {
        "model_state_dict": model.state_dict(), 
        "class_names": idx2class
    },
    "/Users/dwiref/Downloads/inceptionv3_flowers.pt"
)

In [ ]:
# Replace these paths accordingly
preds_manifest = "/Users/dwiref/Downloads/predictions.json"
trained_weights = "/Users/dwiref/Downloads/inceptionv3_flowers.pt"

In [ ]:
# Load model checkpoint for trained weights
ckpt = torch.load(trained_weights, map_location=device)
pred_model = models.inception_v3()
pred_model.fc = torch.nn.Linear(
    pred_model.fc.in_features,
    len(idx2class)
)
pred_model.load_state_dict(ckpt["model_state_dict"])
pred_model.to(device).eval()

In [ ]:
# Transformation to apply before predictions
test_tf = transforms.Compose([
    transforms.Resize(320), transforms.CenterCrop(299),
    transforms.ToTensor(),  transforms.Normalize([0.5]*3, [0.5]*3),
])

In [ ]:
# Run predictions on the list of images from the test split, and save
# the results to a manifest that can be ingested using the FiftyOne importer
# for dataset type FiftyOneImageClassificationDataset
labels = {}
for img_path in test_sample_images:
    img_path = Path(img_path)
    x = test_tf(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x)
        pred_idx = pred.argmax(1).item()
    # key = base filename (no .jpg)
    key = f"{DATASET_DIR}/{img_path.parent.name}/{img_path.name}"
    labels[key] = pred_idx

In [ ]:
manifest_like = {"classes": dataset.default_classes, "labels": labels}
with open(preds_manifest, "w") as f:
    json.dump(manifest_like, f, indent=4)

In [ ]:
# Ingest predictions on the test set back into the original dataset
test_pred_dataset = fo.Dataset.from_dir(
    dataset_type=fo.types.FiftyOneImageClassificationDataset,
    labels_path=preds_manifest,
    name='test-flowers-preds'
)

# Merge prediction results back into the original dataset
dataset.merge_samples(test_pred_dataset)